# Notebook 10: Optimizer & Scheduler

**Module:** 12 Capstone — water-bodies-detection  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Understand AdamW choice
2. Derive WarmupCosine schedule
3. Explain weight decay and grad clipping
4. Tune LR for new datasets

## AdamW

- Adaptive per-parameter learning rates
- Decoupled weight decay (1e-5) — regularizes without fighting momentum
- Better than SGD for transfer learning small datasets

## WarmupCosine Class

```python
# Warmup: linear 0 → base_lr over warmup epochs
# Cosine: base_lr → min_lr over remaining epochs
lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + cos(π * t))
```

**Stage 1:** warmup=2, base=2e-4
**Stage 2:** warmup=3, base=2e-5

## Gradient Clipping

`grad_clip_norm=1.0` — prevents explosion when boundary loss spikes on sparse pixels.

In [ ]:
import os, json, yaml
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
REPO = '../../water-bodies-detection'
print('Capstone repo path:', os.path.abspath(REPO) if os.path.exists(REPO) else 'clone water-bodies-detection alongside Machine-Learning')

In [ ]:
import numpy as np

def warmup_cosine_lr(epoch, base_lr, epochs, warmup, min_lr):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, epochs - warmup)
    t = min(1.0, max(0.0, t))
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * t))

lrs = [warmup_cosine_lr(e, 2e-4, 12, 2, 1e-8) for e in range(12)]
plt.plot(lrs); plt.xlabel('epoch'); plt.ylabel('LR'); plt.title('Stage 1 schedule'); plt.show()

---

## Summary

WarmupCosine + AdamW + grad clip = stable two-stage fine-tuning.

**Next:** [11_Metrics_and_Validation.ipynb](11_Metrics_and_Validation.ipynb)